# EDA - Carga de Energia ONS

O Operador Nacional do Sistema Elétrico (ONS) é a entidade responsável por coordenar e monitorar o sistema elétrico brasileiro, garantindo o equilíbrio entre geração e consumo de energia no país.

A carga energética representa a demanda/consumo de energia elétrica em determinado período, enquanto a unidade MWmed (Megawatt médio) indica a potência média consumida ao longo do tempo analisado.

O sistema elétrico brasileiro é dividido nos subsistemas Norte, Nordeste, Sul e Sudeste/Centro-Oeste, utilizados pelo ONS para organizar e operar o Sistema Interligado Nacional.

O objetivo desta análise exploratória é compreender o comportamento da carga energética brasileira, identificando padrões temporais, diferenças entre subsistemas, dispersões e possíveis outliers a partir dos dados disponibilizados pelo ONS.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/raw/CARGA_ENERGIA_2024.csv", sep=";", encoding="utf-8") #lê o arquivo CSV e armazena em um dataframe
df.head() #mostra as primeiras linhas do dataframe
df.info() #informações sobre o dataframe, como tipos de dados e valores nulos

#padronização dos nomes das colunas:
# df.columns = (
#     df.columns
#     .str.strip() #remove espaços em branco
#     .str.lower() #deixa tudo em minúsculo
#     .str.replace(" ", "_") #substitui espaços por underline
# ) 
print(df.columns.tolist())

In [ ]:
#tratamento de dados:
df["din_instante"] = pd.to_datetime(df["din_instante"]) #converte a coluna "din_instante" para o formato datetime
# df["val_cargaenergiamwmed"] = pd.to_numeric(df["val_cargaenergiamwmed"]) #converte a coluna "val_cargaenergiamwmed" para o formato numérico
#verificar se há valores nulos:
print((df.isnull().sum()))
#não foram encontrados valores nulos, então segue a análise
print(df.duplicated().sum())
#não foram encontrados valores duplicados, então segue a análise
print(df["din_instante"].dtype)

In [ ]:
#extração de informações de data:
df["din_instante_ano"] = df["din_instante"].dt.year
df["din_instante_mes"] = df["din_instante"].dt.month
df["din_instante_dia"] = df["din_instante"].dt.day

df.describe() #detecta outilers e fornece estatísticas descritivas
df = df.sort_values("din_instante") #ordena em series temporais
print(df.head())

In [ ]:
# plt.figure(figsize=(14,5))
# plt.plot(
#     df["din_instante"],
#     df["val_cargaenergiamwmed"]
# )
# plt.title("Carga de Energia ao Longo do Tempo")
# plt.xlabel("Data")
# plt.ylabel("Carga MWmed")
# plt.show()

#Com base no feedback, decidi usar a média móvel para suavizar a série temporal e facilitar a visualização das tendências ao longo do tempo. 
#A média móvel de 7 dias é uma escolha comum para séries temporais diárias, pois ajuda a reduzir o ruído e destacar as tendências subjacentes.

df["media_movel"] = (
    df.groupby("nom_subsistema")["val_cargaenergiamwmed"]
    .transform(lambda x: x.rolling(7).mean())
)

plt.figure(figsize=(14,6))

sns.lineplot(
    data=df,
    x="din_instante",
    y="media_movel",
    hue="nom_subsistema"
)

plt.title("Carga Energética por Subsistema (Média Móvel de 7 dias)")
plt.xlabel("Data")
plt.ylabel("Carga MWmed")

plt.grid()
plt.show()

Com base no arquivo "reports\figures\carga_energetica_por_subsistema(media_movel_de_7_dias).png" gerado a partir da execução da célula acima é possível concluir que :
- há um comportamento sazonal nas séries temporais, com oscilações recorrentes ao longo do tempo.
- o uso da média móvel reduziu ruídos de curto prazo e facilitou a visualização das tendências gerais de consumo energético nos diferentes subsistemas.
- o subsistema Sudeste/Centro-Oeste apresentou os maiores níveis de carga energética ao longo da análise, comportamento que pode ser associado à elevada concentração populacional e industrial da região 
- as diferenças observadas entre os subsistemas podem estar relacionadas a fatores econômicos, populacionais e industriais de cada região brasileira.

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=df,
    x="nom_subsistema",
    y="val_cargaenergiamwmed"
)

plt.title("Distribuição da Carga por Subsistema")
plt.xlabel("Subsistema")
plt.ylabel("Carga MWmed")

plt.show()

A partir do arquivo "distribuicao_da_carga_por_subsistema.png" se obteve as seguintes conclusões:
- foi visto que o subsistema SE apresenta maior mediana de carga de consumo com valor mínimo mais elevadoo que se alinha no fato de ser a região do país com a maior densidade demográfica e com a maior concentração de industrias indicando a possibilidade de maior demanda energética
- o subsistema N teve os menores valores além de serem os mais concentrados possivelmenre relacionado com o dado de ser a região com a menor densidade populacional e diferentes demandas energéticas da população por ser uma região de clima equatorial
- os subsistemas NE e S têm valores similares com o subsistema S apresentando maior variação de valores além de outliers que indicam cargas excepcionalmente altas 
- em todos os subsistemas foi analisado uma mediana bem centralizada e bloxplots relativamente simétricos com variação de dados mais concentradas 

In [ ]:
media_mensal = (
    df.groupby("din_instante_mes")["val_cargaenergiamwmed"]
    .mean()
)

plt.figure(figsize=(10,5))

media_mensal.plot(marker="o")

plt.title("Média Mensal da Carga Energética")
plt.xlabel("Mês")
plt.ylabel("Carga Média MWmed")

plt.grid()

plt.show()

A partir da observação do gráfico do arquivo "media_mensal_da_carga_energetica.png" é notório que a média mensal da carga energética possui uma sazonalidade, onde, os meses inicias do ano possuem uma média relativa mais alta enquanto os meses medianos apresentam uma tendência de queda na média, que volta a subir nos meses finais.

In [ ]:
plt.figure(figsize=(12,4))

sns.boxplot(
    x=df["val_cargaenergiamwmed"]
)

plt.title("Identificação de Outliers")

plt.show()

O gráfico do arquivo "identificacao_de_outliers.png" mostra um boxplot de valores globais, ou seja, considera todas as cargas de todos os meses. Com isso:
- é observado valores extremos acima dos valores padrão indicando picos de consumo 
- ademais, é visto que há um alongamento do terceiro quadrante indicando maior variação dos valores acima do padrão
- essas alterações podem ser em virtude de alguns fatores: 
    - alterações climáticas
    - sazonlidade
    - maior demanda desse recurso
    - alteração temporaria de demanda

# Conclusões

A análise da carga energética por subsistema utilizando média móvel de 7 dias permitiu identificar comportamento sazonal nas séries temporais, além de reduzir ruídos de curto prazo e destacar tendências gerais de consumo.

O subsistema Sudeste/Centro-Oeste apresentou os maiores níveis de carga energética ao longo do período analisado, comportamento possivelmente associado à maior concentração populacional e industrial da região.

Também foram observadas diferenças relevantes entre os subsistemas, que podem estar relacionadas a fatores econômicos, populacionais e industriais de cada região brasileira.

A análise dos boxplots permitiu identificar a presença de outliers em alguns subsistemas, representando valores de carga energética significativamente acima do comportamento padrão observado nos dados. Esses valores extremos podem estar associados a períodos específicos de alta demanda energética, influenciados por fatores como condições climáticas, aumento do consumo residencial, atividade industrial elevada ou eventos sazonais.

Entretanto, os outliers identificados não necessariamente representam erros nos dados, podendo refletir comportamentos reais e relevantes do sistema elétrico brasileiro.